In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Performance Management: Uma Função Qiskit pela Q-CTRL Fire Opal
*Veja a [referência de API](https://docs.quantum.ibm.com/api/functions/q-ctrl-performance-management)*

> **Note:** As Funções Qiskit são um recurso experimental disponível apenas para usuários dos planos IBM Quantum&reg; Premium Plan, Flex Plan e On-Prem (via API da IBM Quantum Platform). Elas estão em status de versão prévia e sujeitas a mudanças.


<Accordion>
<AccordionItem title="Package versions">

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit[all]~=2.3.1
qiskit-ibm-runtime~=0.45.1
```
</AccordionItem>
</Accordion>
## Visão geral
O Fire Opal Performance Management torna simples para qualquer pessoa obter resultados significativos de computadores quânticos em escala, sem precisar ser especialista em hardware quântico. Ao executar circuitos com o Fire Opal Performance Management, técnicas de supressão de erros baseadas em IA são aplicadas automaticamente, permitindo o escalonamento de problemas maiores com mais gates e qubits. Essa abordagem reduz o número de shots necessários para chegar à resposta correta, sem nenhuma sobrecarga adicional — resultando em economias significativas em tempo de computação e custo.

O Performance Management suprime erros e aumenta a probabilidade de obter a resposta correta em hardware com ruído. Em outras palavras, ele aumenta a relação sinal-ruído. A imagem a seguir mostra como a maior precisão proporcionada pelo Performance Management pode reduzir a necessidade de shots adicionais no caso de um algoritmo de Transformada de Fourier Quântica de 10 qubits. Com apenas 30 shots, a Q-CTRL atinge o limiar de confiança de 99%, enquanto o padrão (`QiskitRuntime` Sampler, `optimization_level`=3 e `resilience_level`=1, `ibm_sherbrooke`) requer 170.000 shots. Ao obter a resposta correta mais rapidamente, você economiza tempo de computação significativo.

![Visualização do tempo de execução melhorado](../docs/images/guides/qctrl-performance-management/achieve_more.svg)

A função Performance Management pode ser usada com qualquer algoritmo, e você pode facilmente utilizá-la no lugar das [primitivas padrão do Qiskit Runtime](/guides/primitives). Nos bastidores, múltiplas técnicas de supressão de erros trabalham em conjunto para evitar que erros ocorram em tempo de execução. Todos os métodos de pipeline do Fire Opal são pré-configurados e agnósticos ao algoritmo, o que significa que você sempre obtém o melhor desempenho logo de início.

Para obter acesso ao Performance Management, [entre em contato com a Q-CTRL](https://form.typeform.com/to/uOAVDnGg?typeform-source=q-ctrl.com).
## Descrição
O Fire Opal Performance Management possui duas opções de execução semelhantes às primitivas do Qiskit Runtime, para que você possa facilmente substituir pelo Sampler e Estimator da Q-CTRL. O fluxo de trabalho geral para usar a função Performance Management é:
1. Defina seu circuito (e operadores no caso do Estimator).
2. Execute o circuito.
3. Recupere os resultados.

Para reduzir o ruído do hardware, o Fire Opal emprega uma série de técnicas de supressão de erros baseadas em IA, descritas na imagem a seguir. Com o Fire Opal, todo o pipeline é completamente automatizado, sem necessidade de configuração.

O pipeline do Fire Opal elimina a necessidade de sobrecarga adicional, como aumento do tempo de execução quântica ou qubits físicos extras. Observe que o tempo de processamento clássico continua sendo um fator (consulte a seção [Benchmarks](#benchmarks) para estimativas, onde "Tempo total" reflete tanto o processamento clássico quanto o quântico). Ao contrário da mitigação de erros, que requer sobrecarga na forma de amostragem, a supressão de erros do Fire Opal funciona nos níveis de gate e pulso para lidar com várias fontes de ruído e prevenir a ocorrência de erros. Ao prevenir erros, a necessidade de pós-processamento custoso é eliminada.

A imagem a seguir mostra os métodos de supressão de erros automatizados pelo Fire Opal Performance Management.

![Visualização do pipeline de supressão de erros](../docs/images/guides/qctrl-performance-management/error_suppression.svg)

A função oferece duas primitivas, Sampler e Estimator, e as entradas e saídas de ambas estendem a especificação implementada para as [primitivas Qiskit Runtime V2](/guides/primitive-input-output#pubs).
## Benchmarks
Os resultados de [benchmarking algorítmico publicados](https://journals.aps.org/prapplied/abstract/10.1103/PhysRevApplied.20.024034) demonstram melhora significativa de desempenho em vários algoritmos, incluindo Bernstein-Vazirani, transformada de Fourier quântica, busca de Grover, algoritmo de otimização aproximada quântica e eigensolver variacional quântico. O restante desta seção fornece mais detalhes sobre os tipos de algoritmos que você pode executar, bem como o desempenho e os tempos de execução esperados.

Os seguintes estudos independentes demonstram como o Performance Management da Q-CTRL viabiliza pesquisa algorítmica em escala recorde:
- [Parametrized Energy-Efficient Quantum Kernels for Network Service Fault Diagnosis](https://arxiv.org/abs/2405.09724v1) - aprendizado de kernel quântico com até 50 qubits
- [Tensor-based quantum phase difference estimation for large-scale demonstration](https://arxiv.org/abs/2408.04946) - estimativa de fase quântica com até 33 qubits
- [Hierarchical Learning for Quantum ML: Novel Training Technique for Large-Scale Variational Quantum Circuits](https://arxiv.org/abs/2311.12929) - carregamento de dados quânticos com até 21 qubits

A tabela a seguir fornece um guia aproximado de precisão e tempos de execução de execuções de benchmarking anteriores no `ibm_fez`. O desempenho em outros dispositivos pode variar. O tempo de uso é baseado em uma suposição de 10.000 shots por circuito. O "Número de qubits" indicado não é uma limitação rígida, mas representa limiares aproximados onde você pode esperar precisão de solução extremamente consistente. Problemas maiores foram resolvidos com sucesso, e testes além desses limites são encorajados.

| Exemplo    | Número de qubits | Precisão | Medida de precisão | Tempo total (s) | Uso de runtime (s) | Primitiva (Modo) |
| ---------  | ---------------- | -------------------------- | -------- | ---------- | ------------- |------------- |
| Bernstein–Vazirani  |  50Q    | 100%  | Taxa de Sucesso (Porcentagem de execuções em que a resposta correta é a bitstring com maior contagem)     | 10    | 8         | Sampler |
| Transformada de Fourier Quântica | 30Q              | 100% | Taxa de Sucesso (Porcentagem de execuções em que a resposta correta é a bitstring com maior contagem)      | 10    | 8        | Sampler |
| Estimativa de Fase Quântica  | 30Q   | 99,9998%  | Precisão do ângulo encontrado: `1- abs(real_angle - angle_found)/pi`  | 10  | 8  | Sampler |
| Simulação quântica: Modelo de Ising (15 passos) | 20Q  | 99,775%   |  $A$ (definido abaixo)  |  60 (por passo)  | 15 (por passo)   | Estimator |
| Simulação quântica 2: dinâmica molecular (20 pontos de tempo) | 34Q  |  96,78%  |  $A_{mean}$ (definido abaixo)   | 10 (por ponto de tempo)    | 6 (por ponto de tempo)  | Estimator |

Definindo a precisão da medição de um valor esperado - a métrica $A$ é definida como segue:
$$
A = 1 - \frac{|\epsilon^{ideal} - \epsilon^{meas}|}{\epsilon^{ideal}_{max} - \epsilon^{ideal}_{min}},
$$
onde $ \epsilon^{ideal} $ = valor esperado ideal,  $ \epsilon^{meas} $ = valor esperado medido, $\epsilon^{ideal}_{max} $ = valor máximo ideal, e $\epsilon^{ideal}_{min}$ = valor mínimo ideal. $A_{mean}$ é simplesmente a média do valor de $A$ ao longo de múltiplas medições.

Essa métrica é usada porque é invariante a deslocamentos globais e escalonamento no intervalo de valores atingíveis. Em outras palavras, independentemente de você deslocar o intervalo de possíveis valores esperados para cima ou para baixo, ou aumentar a dispersão, o valor de $A$ deve permanecer consistente.
## Primeiros passos
O Fire Opal Performance Management usa Qiskit v`2.0.0`, que é a versão recomendada. As versões suportadas são Qiskit >=v`2.0.0`.
Autentique-se usando sua [chave de API da IBM Quantum Platform](http://quantum.cloud.ibm.com/) e selecione a Função Qiskit conforme mostrado a seguir. (Este trecho assume que você já [salvou sua conta](/guides/functions#install-qiskit-functions-catalog-client) no seu ambiente local.)

In [1]:
# This cell is hidden from users. It hides the "...You have imported samplomatic..." warning.
import warnings

warnings.filterwarnings("ignore", message=".*You have imported samplomatic*")

In [2]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# verify that you have access to the function
catalog.list()

[QiskitFunction(qunova/hivqe-chemistry),
 QiskitFunction(global-data-quantum/quantum-portfolio-optimizer),
 QiskitFunction(algorithmiq/tem),
 QiskitFunction(qedma/qesem),
 QiskitFunction(multiverse/singularity),
 QiskitFunction(ibm/circuit-function),
 QiskitFunction(q-ctrl/optimization-solver),
 QiskitFunction(colibritd/quick-pde),
 QiskitFunction(q-ctrl/performance-management),
 QiskitFunction(kipu-quantum/iskay-quantum-optimizer)]

In [3]:
# Access Function
perf_mgmt = catalog.load("q-ctrl/performance-management")

<Admonition type="note" title="Does this function support all IBM backends?">
If you want to use a backend that this function does not currently support, [reach out to Q-CTRL](https://form.typeform.com/to/iuujEAEI?typeform-source=q-ctrl.com) to add support.
</Admonition>

## Estimator primitive

### Estimator example

Use Fire Opal Performance Management's Estimator primitive to determine the expectation value of a single circuit-observable pair.

In addition to the `qiskit-ibm-catalog` and `qiskit` packages, you will also use the `numpy` package to run this example. You can install this package by uncommenting the following cell if you are running this example in a notebook using the IPython kernel.

In [4]:
# %pip install numpy

**2. Executar o circuito**

Execute o circuito e, opcionalmente, defina o backend e o número de shots.

In [5]:
import numpy as np
from qiskit.circuit.library import iqp
from qiskit.quantum_info import random_hermitian, SparsePauliOp

n_qubits = 50

# Generate a random circuit
mat = np.real(random_hermitian(n_qubits, seed=1234))
circuit = iqp(mat)
circuit.measure_all()

# Define observables as a string
observable = SparsePauliOp("Z" * n_qubits)

In [6]:
# Create PUB tuple
estimator_pubs = [(circuit, observable)]

Você pode usar as [APIs do Qiskit Serverless](/guides/serverless) familiares para verificar o status da sua carga de trabalho da Função Qiskit:

In [7]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy().name

In [8]:
# Run the circuit using Estimator
qctrl_estimator_job = perf_mgmt.run(
    primitive="estimator",
    pubs=estimator_pubs,
    backend_name=backend_name,
)

**3. Recuperar o resultado**

In [9]:
qctrl_estimator_job.status()

'QUEUED'

Os resultados têm o mesmo formato de um [resultado do Estimator](/guides/estimator-input-output):

In [10]:
# Retrieve the counts from the result list
result = qctrl_estimator_job.result()

The results have the same format as an [Estimator result](/docs/guides/estimator-input-output):

In [22]:
import numpy

result_str = str(result)

with numpy.printoptions(threshold=200):
    print(
        f"The result of the submitted job had {len(result)} PUB "
        f"and has a value:\n {result[0]}\n"
    )

print("The associated PubResult of this job has the following DataBins:")
print(f"{result[0].data}\n")

print(f"And this DataBin has attributes: {result[0].data.keys()}")

print("The expectation values measured from this PUB are:")
print(f"{result[0].data.evs}")

The result of the submitted job had 1 PUB
The result of the submitted job had 1 PUB and has a value:
 PubResult(data=DataBin(evs=0.0195, stds=0.9998098569228051), metadata={'precision': None})

The associated PubResult of this job has the following DataBins:
DataBin(evs=0.0195, stds=0.9998098569228051)

And this DataBin has attributes: dict_keys(['evs', 'stds'])
The expectation values measured from this PUB are:
0.0195


## Primitiva Sampler
### Exemplo com Sampler
Use a primitiva Sampler do Fire Opal Performance Management para executar um circuito Bernstein–Vazirani. Esse algoritmo, usado para encontrar uma string oculta a partir das saídas de uma função de caixa-preta, é um algoritmo de benchmarking comum porque existe uma única resposta correta.
**1. Criar o circuito**

Defina a resposta correta para o algoritmo, a bitstring oculta, e o circuito Bernstein–Vazirani. Você pode ajustar a largura do circuito simplesmente alterando o `circuit_width`.

In [12]:
import qiskit

circuit_width = 35
hidden_bitstring = "1" * circuit_width

# Create circuit, reserving one qubit for BV oracle
bv_circuit = qiskit.QuantumCircuit(circuit_width + 1, circuit_width)
bv_circuit.x(circuit_width)
bv_circuit.h(range(circuit_width + 1))
for input_qubit, bit in enumerate(reversed(hidden_bitstring)):
    if bit == "1":
        bv_circuit.cx(input_qubit, circuit_width)
bv_circuit.barrier()
bv_circuit.h(range(circuit_width + 1))
bv_circuit.barrier()
for input_qubit in range(circuit_width):
    bv_circuit.measure(input_qubit, input_qubit)

# Create PUB tuple
sampler_pubs = [(bv_circuit,)]

**2. Executar o circuito**

Execute o circuito e, opcionalmente, defina o backend e o número de shots.

In [13]:
# Run the circuit using Sampler
qctrl_sampler_job = perf_mgmt.run(
    primitive="sampler",
    pubs=sampler_pubs,
    backend_name=backend_name,
)

Verifique o [status](/guides/functions#check-job-status) da sua carga de trabalho da Função Qiskit ou recupere os [resultados](/guides/functions#retrieve-results) da seguinte forma:

In [14]:
# Print the ID so you can use it later, if necessary
print(qctrl_sampler_job.job_id)

qctrl_sampler_job.status()

60fe2fa1-a860-43e4-8615-c6ac4180f93b


'QUEUED'

**3. Retrieve the result**

In [15]:
# Retrieve the job results
sampler_result = qctrl_sampler_job.result()

In [16]:
# Get results for the first (and only) PUB
pub_result = sampler_result[0]
counts = pub_result.data.c.get_counts()

print("Counts for the meas output register (limited to 30 results):")
for i, (bitstring, count) in enumerate(counts.items()):
    if i >= 50:
        print(f"  ... ({len(counts) - 30} more items)")
        break
    print(f"  {bitstring}: {count}")

Counts for the meas output register (limited to 30 results):
  11111111111111111111111111111111111: 1661
  11111111111111111111111111110111111: 60
  11111111111111111111111111111101111: 54
  11111111111111111111111111111110111: 54
  11111111111111011111111111111111111: 46
  11111111111111111110111111111111111: 44
  11111111111111111111111101111111111: 42
  11111111111111111111111110111111111: 42
  11111111111111110111111111111111111: 41
  11111111111111111111111111111111101: 39
  11111111111111111111101111111111111: 38
  11111111111111111111110111111111111: 38
  11111111111111111111111111101111111: 37
  11111111111111111111111111111111110: 36
  11111111111110111111111111111111111: 35
  11111111111111111111111111111011111: 32
  11111111111111101111111111111111111: 32
  01111111111111111111111111111111111: 27
  11111111111111111011111111111111111: 23
  11111111101111111111111111111111111: 22
  11111111111111111111111111111111011: 21
  11111111011111111111111111111111111: 20
  00000000000

**3. Plot the top bitstrings**

Plot the bitstring with the highest counts to see if the hidden bitstring was the mode.

In [17]:
import matplotlib.pyplot as plt


def plot_top_bitstrings(counts_dict, hidden_bitstring=None):
    # Sort and take the top 100 bitstrings
    top_100 = sorted(counts_dict.items(), key=lambda x: x[1], reverse=True)[
        :100
    ]
    if not top_100:
        print("No bitstrings found in the input dictionary.")
        return

    # Unzip the bitstrings and their counts
    bitstrings, counts = zip(*top_100)

    # Assign colors: purple if the bitstring matches hidden_bitstring,
    # otherwise gray
    colors = [
        "#680CE9" if bit == hidden_bitstring else "gray" for bit in bitstrings
    ]

    # Create the bar plot
    plt.figure(figsize=(15, 8))
    plt.bar(
        range(len(bitstrings)), counts, tick_label=bitstrings, color=colors
    )

    # Rotate the bitstrings for better readability
    plt.xticks(rotation=90, fontsize=8)
    plt.xlabel("Bitstrings")
    plt.ylabel("Counts")
    plt.title("Top 100 Bitstrings by Counts")

    # Show the plot
    plt.tight_layout()
    plt.show()

The hidden bitstring is highlighted in purple, and it should be the bitstring with the highest number of counts.

In [18]:
plot_top_bitstrings(counts, hidden_bitstring)

<Image src="../docs/images/guides/q-ctrl-performance-management/extracted-outputs/8106d906-0.avif" alt="Output of the previous code cell" />

**3. Plotar as principais bitstrings**

Plote a bitstring com o maior número de contagens para ver se a bitstring oculta foi a moda.